# U-Net34 + Optuna đầy đủ trên Kaggle — 1 GPU

Notebook tự chứa toàn bộ pipeline và đọc trực tiếp dataset Kaggle [`tinlngnguyn/skin-lesion-segmentation-hus`](https://www.kaggle.com/datasets/tinlngnguyn/skin-lesion-segmentation-hus) tại `/kaggle/input/skin-lesion-segmentation-hus`. Ảnh được color balance bằng đúng `simplest_cb(..., 1)` từ `fredguth/skin-cancer/color_balance.py` trước khi resize.

Quy trình mặc định:

1. Kiểm tra nghiêm ngặt 2.594 train, 100 validation và 1.000 test cùng cặp image–mask.
2. Chia **official train** một lần thành train subset 80% và internal-validation subset 20%.
3. Optuna ở 128 px: TPE + percentile pruning theo từng epoch, xếp hạng bằng Best Dice trên internal validation.
4. Giữ Top-5 rồi tiếp tục huấn luyện từng candidate ở 256 px trên đúng split nội bộ để khóa winner.
5. Khởi tạo lại ImageNet U-Net34 và train winner bằng 3-fold trên toàn bộ official train, theo chuỗi 128 frozen/unfrozen → 256 frozen/unfrozen. Sau mỗi stage, model chỉ đi tiếp nếu đạt global Best Dice tốt hơn; nếu không, checkpoint tốt nhất trước đó được khôi phục.
6. Sau khi mọi lựa chọn đã khóa, đánh giá từng fold và ensemble trên official validation; đánh giá single model đã chọn trước và ensemble trên official test.

Không có official validation/test nào được đưa vào LR Finder, augmentation, pruning, chọn trial, chọn checkpoint, chọn threshold hay chọn ensemble weight. Chỉ train subset/fold-train có augmentation; mọi validation/holdout/test đều dùng transform sạch.

> Kaggle Settings: Attach dataset trên, chọn **GPU** và bật Internet ít nhất ở lần đầu để tải ResNet34 ImageNet weights (và cài Optuna nếu Kaggle image chưa có). Pipeline cố định ở `cuda:0`, kể cả runtime hiển thị nhiều GPU. Full run có thể dài hơn một phiên Kaggle; dùng `RESTORE_FROM` trong cell cấu hình để tiếp tục từ output của phiên trước.


In [ ]:
# Chỉ cài Optuna khi Kaggle image chưa có; không nâng cấp torch/torchvision giữa session.
import importlib.util
import subprocess
import sys

if importlib.util.find_spec('optuna') is None:
    print('Optuna chưa có; đang cài optuna>=4,<5 ...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'optuna>=4,<5'])
else:
    print('Optuna đã có sẵn trong Kaggle runtime.')


In [ ]:
import gc
import hashlib
import json
import math
import os
import pickle
import random
import shutil
import time
from pathlib import Path

os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
# Phải đặt trước khi import torch: kể cả Kaggle bật T4x2, process chỉ nhìn thấy GPU đầu.
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import albumentations as A
import cv2
import numpy as np
import optuna
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from albumentations.pytorch import ToTensorV2
from IPython.display import display
from PIL import Image
from scipy import ndimage
from torch.utils.data import DataLoader, Dataset, get_worker_info
from torchvision import models
from tqdm.auto import tqdm

try:
    import psutil
except ImportError:
    psutil = None

# -------------------- Điều khiển pipeline --------------------
FAST_DEV_RUN = False          # True: test nhanh toàn flow bằng subset/ít epoch.
RUN_HPO = True                # Optuna @128.
RUN_CONFIRM_256 = True        # Top-K @128 -> confirmation @256.
RUN_FINAL_CV = True           # Winner -> final 3-fold 128/256.
RUN_OFFICIAL_EVALUATION = True

# Để resume qua session: attach output phiên trước làm Kaggle Input rồi đặt đường dẫn
# tới đúng thư mục full_v2_color_balance/fast_dev_v2_color_balance, ví dụ:
# RESTORE_FROM = Path('/kaggle/input/my-previous-run/full_v2_color_balance')
RESTORE_FROM = None

DATASET_HANDLE = 'tinlngnguyn/skin-lesion-segmentation-hus'
DATA_ROOT = Path('/kaggle/input/skin-lesion-segmentation-hus')
SPLIT_DIRECTORIES = {
    'train': ('ISIC2018_Task1-2_Training_Input', 'ISIC2018_Task1_Training_GroundTruth'),
    'validation': ('ISIC2018_Task1-2_Validation_Input', 'ISIC2018_Task1_Validation_GroundTruth'),
    'test': ('ISIC2018_Task1-2_Test_Input', 'ISIC2018_Task1_Test_GroundTruth'),
}
EXPECTED_COUNTS = {'train': 2594, 'validation': 100, 'test': 1000}

SEED = 42
HPO_VAL_FRACTION = 0.20
TARGET_TOTAL_TRIALS = 30       # Tổng số trial terminal mong muốn, không phải số trial cộng thêm.
TOP_K = 5
N_FINAL_FOLDS = 3
OPTUNA_TIMEOUT_SECONDS = None  # Ví dụ 10*3600; None = chạy đến đủ TARGET_TOTAL_TRIALS.
# Với direction=maximize, 25 giữ nhóm top ~25% ở cùng step (pruning khá mạnh).
# Nếu chỉ muốn loại bottom ~25%, đổi thành 75.0 trước lần chạy đầu tiên.
PRUNER_PERCENTILE = 75.0

HPO_FROZEN_EPOCHS = 30
HPO_UNFROZEN_EPOCHS = 30
CONFIRM_FROZEN_EPOCHS = 30
CONFIRM_UNFROZEN_EPOCHS = 30
FINAL_FROZEN_EPOCHS = 30
FINAL_UNFROZEN_EPOCHS = 30

SEARCH_BATCHES_128 = (8, 16)
MAX_BATCH_256 = 8             # 1 GPU T4: không dùng batch 16 ở 256 px.
NUM_WORKERS = min(2, os.cpu_count() or 1)
PREFETCH_FACTOR = 1
PIN_MEMORY = True
CACHE_RESIZED_TRAIN_IN_RAM = True
DROP_OS_PAGE_CACHE = True
SAVE_LAST_EVERY = 5

LRF_ITERS = 50
LR_MIN, LR_MAX, LRF_BETA = 1e-6, 1.0, 0.9
LR_SELECTION_FACTOR = 0.1
MAX_LR_FROZEN = {'SGD': 1e-1, 'Adam': 3e-3, 'AdamW': 3e-3}
MAX_LR_UNFROZEN = {'SGD': 3e-2, 'Adam': 1e-3, 'AdamW': 1e-3}
CUT_FRAC, STLR_RATIO = 0.1, 32.0
GRAD_CLIP_NORM = 1.0
PRED_THRESHOLD = 0.5
JACCARD_CUT = 0.65
PRETRAINED = True
USE_AMP = True
USE_CHANNELS_LAST = False
FP32_TRANSPOSED_CONV = True
PROTOCOL_VERSION = 3
COLOR_BALANCE_PERCENT = 1.0
COLOR_BALANCE_SOURCE = 'fredguth/skin-cancer@master/color_balance.py'

if FAST_DEV_RUN:
    TARGET_TOTAL_TRIALS = 2
    TOP_K = 2
    N_FINAL_FOLDS = 2
    HPO_FROZEN_EPOCHS = HPO_UNFROZEN_EPOCHS = 1
    CONFIRM_FROZEN_EPOCHS = CONFIRM_UNFROZEN_EPOCHS = 1
    FINAL_FROZEN_EPOCHS = FINAL_UNFROZEN_EPOCHS = 1
    LRF_ITERS = 8

if not hasattr(A, 'D4'):
    raise RuntimeError('Albumentations quá cũ: notebook cần A.D4 (Albumentations 2.x).')
try:
    A.Compose([], seed=SEED)
except TypeError as error:
    raise RuntimeError('Albumentations hiện tại không hỗ trợ Compose(seed=...).') from error

OUTPUT_ROOT = Path('/kaggle/working/unet34_optuna_1gpu')
RUN_NAME = ('fast_dev_v2_color_balance' if FAST_DEV_RUN
            else 'full_v2_color_balance')
RUN_DIR = OUTPUT_ROOT / RUN_NAME
STUDY_DB = RUN_DIR / 'optuna.db'
SAMPLER_PATH = RUN_DIR / 'tpe_sampler.pkl'
TOPK_DIR = RUN_DIR / 'hpo_topk_128'
CONFIRM_DIR = RUN_DIR / 'confirmation_256'
FINAL_DIR = RUN_DIR / 'final_cv'
METRICS_DIR = RUN_DIR / 'metrics'
for directory in (RUN_DIR, TOPK_DIR, CONFIRM_DIR, FINAL_DIR, METRICS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

def restore_missing_artifacts(source_root, destination_root):
    source_root, destination_root = Path(source_root), Path(destination_root)
    if not source_root.is_dir():
        raise FileNotFoundError(f'RESTORE_FROM không tồn tại: {source_root}')
    copied = 0
    for source in source_root.rglob('*'):
        relative = source.relative_to(source_root)
        destination = destination_root / relative
        if source.is_dir():
            destination.mkdir(parents=True, exist_ok=True)
        elif not destination.exists():
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, destination)
            copied += 1
    print(f'Restored {copied} file(s) từ {source_root}')

if RESTORE_FROM is not None:
    restore_missing_artifacts(RESTORE_FROM, RUN_DIR)

if not torch.cuda.is_available():
    raise RuntimeError('Hãy bật Accelerator GPU trong Kaggle Settings.')
DEVICE = torch.device('cuda:0')

def set_seed(seed=SEED):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)  # CUDA_VISIBLE_DEVICES chỉ còn đúng cuda:0.
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

set_seed()
cv2.setNumThreads(0)
torch.set_float32_matmul_precision('high')

def memory_status():
    parts = []
    if psutil is not None:
        process = psutil.Process(os.getpid())
        info = process.memory_full_info()
        main_bytes = getattr(info, 'uss', info.rss)
        worker_bytes = 0
        for child in process.children(recursive=True):
            try:
                child_info = child.memory_full_info()
                worker_bytes += getattr(child_info, 'uss', child_info.rss)
            except psutil.Error:
                pass
        parts += [f'ram_main={main_bytes / 2**30:.2f}GiB',
                  f'ram_workers={worker_bytes / 2**30:.2f}GiB']
    parts += [f'cuda_alloc={torch.cuda.memory_allocated(0) / 2**30:.2f}GiB',
              f'cuda_reserved={torch.cuda.memory_reserved(0) / 2**30:.2f}GiB']
    return ' '.join(parts)

print({
    'device_used': str(DEVICE),
    'visible_gpus': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
    'optuna': optuna.__version__,
    'workers': NUM_WORKERS,
    'run_dir': str(RUN_DIR),
})


In [ ]:
IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png'}

def find_populated_directory(directory_name, suffixes):
    outer = DATA_ROOT / directory_name
    candidates = [outer / directory_name, outer]
    if DATA_ROOT.is_dir():
        candidates.extend(path for path in DATA_ROOT.rglob(directory_name) if path.is_dir())
    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen or not candidate.is_dir():
            continue
        seen.add(candidate)
        if any(path.is_file() and path.suffix.lower() in suffixes
               for path in candidate.iterdir()):
            return candidate
    raise FileNotFoundError(
        f'Không tìm thấy thư mục {directory_name} dưới {DATA_ROOT}. '
        f'Hãy Attach dataset {DATASET_HANDLE}.'
    )

def discover_records(split):
    image_name, mask_name = SPLIT_DIRECTORIES[split]
    image_dir = find_populated_directory(image_name, IMAGE_SUFFIXES)
    mask_dir = find_populated_directory(mask_name, {'.png'})
    image_paths = [p for p in image_dir.iterdir()
                   if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES]
    mask_paths = [p for p in mask_dir.iterdir()
                  if p.is_file() and p.suffix.lower() == '.png'
                  and p.stem.endswith('_segmentation')]
    image_map = {p.stem: p for p in image_paths}
    mask_map = {p.stem.removesuffix('_segmentation'): p for p in mask_paths}
    if len(image_map) != len(image_paths):
        raise RuntimeError(f'{split}: trùng image ID')
    if len(mask_map) != len(mask_paths):
        raise RuntimeError(f'{split}: trùng mask ID')
    missing_masks = sorted(set(image_map) - set(mask_map))
    orphan_masks = sorted(set(mask_map) - set(image_map))
    if missing_masks or orphan_masks:
        raise RuntimeError(
            f'{split}: thiếu mask={missing_masks[:5]}, mask mồ côi={orphan_masks[:5]}'
        )
    records = [
        {'image_id': image_id, 'image_path': image_map[image_id],
         'mask_path': mask_map[image_id]}
        for image_id in sorted(image_map)
    ]
    if len(records) != EXPECTED_COUNTS[split]:
        raise RuntimeError(
            f'{split}: cần đúng {EXPECTED_COUNTS[split]} cặp, tìm thấy {len(records)}'
        )
    return records, image_dir, mask_dir

def deterministic_subset(records, count, seed):
    permutation = np.random.default_rng(seed).permutation(len(records))
    return [records[int(i)] for i in permutation[:min(count, len(records))]]

def split_records(records, val_fraction, seed):
    if not 0 < val_fraction < 1:
        raise ValueError('val_fraction phải nằm trong (0, 1)')
    permutation = torch.randperm(
        len(records), generator=torch.Generator().manual_seed(int(seed))
    ).tolist()
    val_count = min(max(1, round(len(records) * val_fraction)), len(records) - 1)
    val_indices = set(permutation[:val_count])
    train_part = [record for i, record in enumerate(records) if i not in val_indices]
    val_part = [record for i, record in enumerate(records) if i in val_indices]
    return train_part, val_part

def ids_digest(records):
    value = '\n'.join(record['image_id'] for record in records)
    return hashlib.sha256(value.encode('utf-8')).hexdigest()

if not DATA_ROOT.is_dir():
    raise FileNotFoundError(
        f'Chưa Attach dataset {DATASET_HANDLE}; không thấy {DATA_ROOT}'
    )

train_records_all, train_image_dir, train_mask_dir = discover_records('train')
official_val_all, val_image_dir, val_mask_dir = discover_records('validation')
official_test_all, test_image_dir, test_mask_dir = discover_records('test')

split_ids = {
    'train': {r['image_id'] for r in train_records_all},
    'validation': {r['image_id'] for r in official_val_all},
    'test': {r['image_id'] for r in official_test_all},
}
assert split_ids['train'].isdisjoint(split_ids['validation'])
assert split_ids['train'].isdisjoint(split_ids['test'])
assert split_ids['validation'].isdisjoint(split_ids['test'])

if FAST_DEV_RUN:
    train_records = deterministic_subset(train_records_all, 96, SEED)
    official_val_records = deterministic_subset(official_val_all, 24, SEED + 1)
    official_test_records = deterministic_subset(official_test_all, 24, SEED + 2)
else:
    train_records = train_records_all
    official_val_records = official_val_all
    official_test_records = official_test_all

hpo_train_records, hpo_val_records = split_records(
    train_records, HPO_VAL_FRACTION, SEED
)
assert {r['image_id'] for r in hpo_train_records}.isdisjoint(
    r['image_id'] for r in hpo_val_records
)
assert len(hpo_train_records) + len(hpo_val_records) == len(train_records)

print('records:', {
    'official_train': len(train_records),
    'hpo_train_subset': len(hpo_train_records),
    'hpo_internal_validation': len(hpo_val_records),
    'official_validation': len(official_val_records),
    'official_test': len(official_test_records),
})
print('train dirs:', train_image_dir, train_mask_dir, sep='\n  ')
print('validation dirs:', val_image_dir, val_mask_dir, sep='\n  ')
print('test dirs:', test_image_dir, test_mask_dir, sep='\n  ')


In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
RAM_CACHE = {}

def apply_mask(matrix, mask, fill_value):
    masked = np.ma.array(matrix, mask=mask, fill_value=fill_value)
    return masked.filled()

def apply_threshold(matrix, low_value, high_value):
    low_mask = matrix < low_value
    matrix = apply_mask(matrix, low_mask, low_value)
    high_mask = matrix > high_value
    return apply_mask(matrix, high_mask, high_value)

def simplest_cb(img, percent):
    """Thuật toán từ fredguth/skin-cancer/color_balance.py."""
    assert img.shape[2] == 3
    assert percent > 0 and percent < 100
    half_percent = percent / 200.0
    out_channels = []
    for channel in cv2.split(img):
        assert len(channel.shape) == 2
        height, width = channel.shape
        flat = np.sort(channel.reshape(width * height))
        low_val = flat[math.floor(flat.shape[0] * half_percent)]
        high_val = flat[math.ceil(flat.shape[0] * (1.0 - half_percent))]
        thresholded = apply_threshold(channel, low_val, high_val)
        normalized = cv2.normalize(
            thresholded, thresholded.copy(), 0, 255, cv2.NORM_MINMAX
        )
        out_channels.append(normalized)
    return cv2.merge(out_channels)

def load_color_balanced_rgb(path):
    # Bản gốc dùng cv2.imread (BGR): balance trong BGR, sau đó mới đổi RGB.
    image_bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if image_bgr is None:
        raise FileNotFoundError(f'OpenCV không đọc được ảnh: {path}')
    if (DROP_OS_PAGE_CACHE and hasattr(os, 'posix_fadvise')
            and hasattr(os, 'POSIX_FADV_DONTNEED')):
        try:
            with open(path, 'rb') as handle:
                os.posix_fadvise(handle.fileno(), 0, 0, os.POSIX_FADV_DONTNEED)
        except OSError:
            pass
    balanced_bgr = simplest_cb(image_bgr, COLOR_BALANCE_PERCENT)
    balanced_rgb = cv2.cvtColor(balanced_bgr, cv2.COLOR_BGR2RGB)
    return balanced_rgb.astype(np.uint8, copy=False)

def load_pil_uint8(path, mode):
    with open(path, 'rb') as handle:
        with Image.open(handle) as source:
            array = np.asarray(source.convert(mode), dtype=np.uint8).copy()
        if (DROP_OS_PAGE_CACHE and hasattr(os, 'posix_fadvise')
                and hasattr(os, 'POSIX_FADV_DONTNEED')):
            try:
                os.posix_fadvise(handle.fileno(), 0, 0, os.POSIX_FADV_DONTNEED)
            except OSError:
                pass
    return array

def build_transform(size, train, seed, preprocessed=False):
    steps = []
    if not preprocessed:
        steps += [
            A.Resize(size, size, interpolation=cv2.INTER_LINEAR,
                     mask_interpolation=cv2.INTER_NEAREST),
        ]
    if train:
        steps += [
            A.D4(p=1.0),
            A.Affine(rotate=(-44, 44), scale=(1.0, 1.05),
                     interpolation=cv2.INTER_LINEAR,
                     mask_interpolation=cv2.INTER_NEAREST, p=0.75),
            A.RandomBrightnessContrast(brightness_limit=0.2,
                                       contrast_limit=0.2, p=0.75),
        ]
    steps.append(ToTensorV2())
    return A.Compose(steps, seed=int(seed))

class ISICDataset(Dataset):
    def __init__(self, records, size, train, seed=SEED):
        if not records:
            raise ValueError('Dataset rỗng')
        self.records = list(records)
        self.size = int(size)
        self.train = bool(train)
        cache = RAM_CACHE.get(self.size)
        indices = None if cache is None else [
            cache['id_to_index'].get(record['image_id']) for record in self.records
        ]
        self.cache = cache if indices is not None and all(i is not None for i in indices) else None
        self.cache_indices = indices if self.cache is not None else None
        self.transform = build_transform(
            self.size, self.train, seed, preprocessed=self.cache is not None
        )

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]
        if self.cache is not None:
            cache_index = self.cache_indices[index]
            image_np = self.cache['images'][cache_index].copy()
            mask_np = self.cache['masks'][cache_index].copy()
        else:
            image_np = load_color_balanced_rgb(record['image_path'])
            mask_np = load_pil_uint8(record['mask_path'], 'L')
            if image_np.shape[:2] != mask_np.shape:
                raise ValueError(f"Image/mask lệch kích thước: {record['image_id']}")
            mask_np = (mask_np > 127).astype(np.uint8)
        transformed = self.transform(image=image_np, mask=mask_np)
        image, mask = transformed['image'], transformed['mask']
        if mask.ndim == 2:
            mask = mask.unsqueeze(0)
        return image.contiguous(), (mask > 0).float().contiguous()

def seed_worker(worker_id):
    del worker_id
    worker_seed = torch.initial_seed() % (2**32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)
    worker = get_worker_info()
    if worker is not None and hasattr(worker.dataset.transform, 'set_random_seed'):
        worker.dataset.transform.set_random_seed(worker_seed)

def make_loader(records, size, train, batch_size, seed, drop_last=False):
    kwargs = dict(
        dataset=ISICDataset(records, size, train, seed),
        batch_size=int(batch_size),
        shuffle=bool(train),
        drop_last=bool(drop_last),
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=False,
        worker_init_fn=seed_worker,
        generator=torch.Generator().manual_seed(int(seed)),
    )
    if NUM_WORKERS > 0:
        kwargs['prefetch_factor'] = PREFETCH_FACTOR
    return DataLoader(**kwargs)

def build_ram_caches(records, sizes):
    records = list(records)
    sizes = tuple(dict.fromkeys(int(size) for size in sizes))
    image_ids = [record['image_id'] for record in records]
    if len(image_ids) != len(set(image_ids)):
        raise ValueError('Trùng image_id trong RAM cache')
    caches = {
        size: {
            'images': np.empty((len(records), size, size, 3), dtype=np.uint8),
            'masks': np.empty((len(records), size, size), dtype=np.uint8),
            'id_to_index': {image_id: i for i, image_id in enumerate(image_ids)},
        }
        for size in sizes
    }
    for index, record in enumerate(tqdm(records, desc=f'RAM cache {sizes}px')):
        image_np = load_color_balanced_rgb(record['image_path'])
        mask_np = load_pil_uint8(record['mask_path'], 'L')
        if image_np.shape[:2] != mask_np.shape:
            raise ValueError(f"Image/mask lệch kích thước: {record['image_id']}")
        binary_mask = (mask_np > 127).astype(np.uint8)
        for size in sizes:
            resized = cv2.resize(image_np, (size, size), interpolation=cv2.INTER_LINEAR)
            caches[size]['images'][index] = resized
            caches[size]['masks'][index] = cv2.resize(
                binary_mask, (size, size), interpolation=cv2.INTER_NEAREST
            )
    return caches

cache_sizes = {128}
if RUN_CONFIRM_256 or RUN_FINAL_CV or RUN_OFFICIAL_EVALUATION:
    cache_sizes.add(256)
if CACHE_RESIZED_TRAIN_IN_RAM:
    RAM_CACHE.update(build_ram_caches(train_records, sorted(cache_sizes)))
    cache_gib = sum(
        value['images'].nbytes + value['masks'].nbytes for value in RAM_CACHE.values()
    ) / 2**30
    print(f'Train-only color-balanced + resized uint8 cache: {cache_gib:.2f} GiB | {memory_status()}')

clean_dataset = ISICDataset(hpo_val_records[:3], 128, train=False, seed=SEED)
for index in range(len(clean_dataset)):
    x1, y1 = clean_dataset[index]
    x2, y2 = clean_dataset[index]
    assert torch.equal(x1, x2) and torch.equal(y1, y2)
    assert x1.dtype == torch.uint8 and y1.dtype == torch.float32
    assert set(torch.unique(y1).tolist()).issubset({0.0, 1.0})
    assert 0.0 < float(y1.mean()) < 1.0
del clean_dataset, x1, x2, y1, y2
print('Dataset, binary mask và clean transform: OK')


In [ ]:
class UpBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, out_channels // 2, 2, stride=2)
        self.skip_projection = nn.Conv2d(skip_channels, out_channels // 2, 1)
        self.bn = nn.BatchNorm2d(out_channels)

    def forward(self, x, skip):
        source_dtype = x.dtype
        if FP32_TRANSPOSED_CONV and x.is_cuda:
            # Guard cho lỗi CUDA illegal memory access đã gặp với ConvTranspose2d FP16.
            with torch.autocast(device_type='cuda', enabled=False):
                x = self.up(x.float())
            x = x.to(dtype=source_dtype)
        else:
            x = self.up(x)
        skip = self.skip_projection(skip)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode='nearest')
        return self.bn(F.relu(torch.cat([x, skip], dim=1)))

class UNet34(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        weights = models.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None
        try:
            backbone = models.resnet34(weights=weights)
        except Exception as error:
            raise RuntimeError(
                'Không tải được ResNet34 ImageNet weights. Hãy bật Internet trong Kaggle.'
            ) from error
        self.stem = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu)
        self.pool = backbone.maxpool
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4
        self.up4 = UpBlock(512, 256, 256)
        self.up3 = UpBlock(256, 128, 256)
        self.up2 = UpBlock(256, 64, 256)
        self.up1 = UpBlock(256, 64, 256)
        self.up0 = UpBlock(256, 3, 16)
        self.head = nn.Conv2d(16, 1, 1)

    @property
    def first_layer_group(self):
        # Property, không đăng ký ModuleList alias nên state_dict không bị nhân đôi key.
        return (self.stem, self.layer1, self.layer2)

    def forward(self, x):
        s0 = self.stem(x)
        s1 = self.layer1(self.pool(s0))
        s2 = self.layer2(s1)
        s3 = self.layer3(s2)
        bottleneck = self.layer4(s3)
        decoded = self.up4(bottleneck, s3)
        decoded = self.up3(decoded, s2)
        decoded = self.up2(decoded, s1)
        decoded = self.up1(decoded, s0)
        decoded = self.up0(decoded, x)
        return self.head(decoded)

def unwrap_model(model):
    return model

def to_runtime(model):
    model = model.to(DEVICE)
    if USE_CHANNELS_LAST:
        model = model.to(memory_format=torch.channels_last)
    return model  # Cố ý chỉ cuda:0, không DataParallel.

def freeze_first_layer_group(model):
    # Giữ đúng semantics dự án: freeze stem/layer1/layer2, các phần còn lại train.
    base = unwrap_model(model)
    for parameter in base.parameters():
        parameter.requires_grad = True
    for module in base.first_layer_group:
        for child in module.modules():
            for parameter in child.parameters(recurse=False):
                parameter.requires_grad = isinstance(child, nn.BatchNorm2d)

def unfreeze_with_frozen_bn(model):
    for parameter in model.parameters():
        parameter.requires_grad = True
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):
            for parameter in module.parameters():
                parameter.requires_grad = False

def set_train_mode(model, freeze_bn):
    model.train()
    if freeze_bn:
        for module in model.modules():
            if isinstance(module, nn.BatchNorm2d):
                module.eval()

def cpu_state_dict(model):
    return {
        key: value.detach().cpu().clone()
        for key, value in unwrap_model(model).state_dict().items()
    }

@torch.no_grad()
def refresh_cpu_state_dict(model, destination):
    source = unwrap_model(model).state_dict()
    if tuple(source) != tuple(destination):
        raise RuntimeError('state_dict keys thay đổi trong lúc train')
    for key, value in source.items():
        destination[key].copy_(value.detach(), non_blocking=False)

def load_model_state(model, state):
    while state and all(key.startswith('module.') for key in state):
        state = {key.removeprefix('module.'): value for key, value in state.items()}
    unwrap_model(model).load_state_dict(state)

shape_model = UNet34(pretrained=False).eval()
with torch.inference_mode():
    shape_output = shape_model(torch.randn(2, 3, 128, 128))
assert shape_output.shape == (2, 1, 128, 128)
assert not any(key.startswith('first_layer_group.') for key in shape_model.state_dict())
print(f'UNet34 parameters: {sum(p.numel() for p in shape_model.parameters()) / 1e6:.3f}M')
del shape_model, shape_output


In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = float(smooth)

    def forward(self, logits, target):
        probability = torch.sigmoid(logits)
        dims = (1, 2, 3)
        intersection = (probability * target).sum(dims)
        denominator = probability.sum(dims) + target.sum(dims)
        dice = (2 * intersection + self.smooth) / (denominator + self.smooth)
        return 1 - dice.mean()

class ComboLoss(nn.Module):
    def __init__(self, alpha=0.5, smooth=1e-6):
        super().__init__()
        self.alpha = float(alpha)
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss(smooth)

    def forward(self, logits, target):
        return self.alpha * self.bce(logits, target) + (1 - self.alpha) * self.dice(logits, target)

class TverskyLoss(nn.Module):
    def __init__(self, alpha=0.5, smooth=1e-6):
        super().__init__()
        self.alpha = float(alpha)
        self.beta = 1 - self.alpha
        self.smooth = float(smooth)

    def forward(self, logits, target):
        probability = torch.sigmoid(logits)
        dims = (1, 2, 3)
        tp = (probability * target).sum(dims)
        fp = (probability * (1 - target)).sum(dims)
        fn = ((1 - probability) * target).sum(dims)
        score = (tp + self.smooth) / (
            tp + self.alpha * fp + self.beta * fn + self.smooth
        )
        return 1 - score.mean()

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = float(gamma)

    def forward(self, logits, target):
        bce = F.binary_cross_entropy_with_logits(logits, target, reduction='none')
        p_t = torch.exp(-bce)
        return (((1 - p_t) ** self.gamma) * bce).mean()

def build_loss(config):
    name = config['name']
    if name == 'BCE':
        return nn.BCEWithLogitsLoss()
    if name == 'Dice':
        return DiceLoss()
    if name == 'Combo':
        return ComboLoss(alpha=config['alpha'])
    if name == 'Tversky':
        return TverskyLoss(alpha=config['alpha'])
    if name == 'Focal':
        return FocalLoss(gamma=config['gamma'])
    raise ValueError(f'Loss không hỗ trợ: {name}')

def sample_configuration(trial):
    batch_size = trial.suggest_categorical('batch_size', list(SEARCH_BATCHES_128))
    optimizer_name = trial.suggest_categorical('optimizer', ['SGD', 'Adam', 'AdamW'])
    optimizer = {'name': optimizer_name}
    if optimizer_name == 'SGD':
        optimizer['momentum'] = trial.suggest_float('momentum', 0.80, 0.99)
        optimizer['weight_decay'] = trial.suggest_float(
            'weight_decay', 1e-6, 1e-3, log=True
        )
    elif optimizer_name == 'Adam':
        optimizer['weight_decay'] = 0.0
    else:
        optimizer['weight_decay'] = trial.suggest_float(
            'weight_decay', 1e-6, 1e-3, log=True
        )

    loss_name = trial.suggest_categorical(
        'loss_type', ['BCE', 'Dice', 'Combo', 'Tversky', 'Focal']
    )
    loss = {'name': loss_name}
    if loss_name in {'Combo', 'Tversky'}:
        loss['alpha'] = trial.suggest_float('alpha', 0.2, 0.8)
    elif loss_name == 'Focal':
        loss['gamma'] = trial.suggest_float('gamma', 1.0, 4.0)
    return {'batch_size': int(batch_size), 'optimizer': optimizer, 'loss': loss}

def configuration_from_params(params):
    optimizer_name = params['optimizer']
    optimizer = {'name': optimizer_name}
    if optimizer_name == 'SGD':
        optimizer.update(momentum=float(params['momentum']),
                         weight_decay=float(params['weight_decay']))
    elif optimizer_name == 'Adam':
        optimizer['weight_decay'] = 0.0
    else:
        optimizer['weight_decay'] = float(params['weight_decay'])
    loss_name = params['loss_type']
    loss = {'name': loss_name}
    if loss_name in {'Combo', 'Tversky'}:
        loss['alpha'] = float(params['alpha'])
    elif loss_name == 'Focal':
        loss['gamma'] = float(params['gamma'])
    return {'batch_size': int(params['batch_size']), 'optimizer': optimizer, 'loss': loss}

def batch_for_size(base_batch_128, size):
    return int(base_batch_128) if int(size) == 128 else min(int(base_batch_128), MAX_BATCH_256)

INPUT_DTYPE = torch.float16 if USE_AMP else torch.float32
GPU_MEAN = torch.tensor(IMAGENET_MEAN, device=DEVICE, dtype=INPUT_DTYPE).view(1, 3, 1, 1)
GPU_STD = torch.tensor(IMAGENET_STD, device=DEVICE, dtype=INPUT_DTYPE).view(1, 3, 1, 1)

def prepare_inputs(inputs):
    inputs = inputs.to(DEVICE, non_blocking=True).to(dtype=INPUT_DTYPE)
    inputs.div_(255.0).sub_(GPU_MEAN).div_(GPU_STD)
    if USE_CHANNELS_LAST:
        return inputs.contiguous(memory_format=torch.channels_last)
    return inputs.contiguous()

def per_image_dice(probability, target, threshold=PRED_THRESHOLD):
    prediction = (probability > threshold).float()
    dims = (1, 2, 3)
    intersection = (prediction * target).sum(dims)
    denominator = prediction.sum(dims) + target.sum(dims)
    return torch.where(
        denominator > 0,
        2 * intersection / denominator.clamp(min=1),
        torch.ones_like(denominator),
    )

@torch.inference_mode()
def evaluate_dice(model, loader):
    model.eval()
    dice_sum = torch.zeros((), device=DEVICE)
    sample_count = 0
    for inputs, targets in loader:
        inputs = prepare_inputs(inputs)
        targets = targets.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_AMP):
            probability = torch.sigmoid(model(inputs).float())
        if not bool(torch.isfinite(probability).all()):
            raise FloatingPointError('Model sinh xác suất NaN/Inf')
        dice_sum += per_image_dice(probability, targets).sum()
        sample_count += targets.shape[0]
    if sample_count == 0:
        raise RuntimeError('Validation loader rỗng')
    return float((dice_sum / sample_count).cpu())

def per_image_scores(probability, target, threshold=PRED_THRESHOLD):
    prediction = (probability > threshold).float()
    dims = (1, 2, 3)
    intersection = (prediction * target).sum(dims)
    prediction_sum = prediction.sum(dims)
    target_sum = target.sum(dims)
    union = prediction_sum + target_sum - intersection
    jaccard = torch.where(union > 0, intersection / union.clamp(min=1),
                          torch.ones_like(union))
    dice = torch.where(prediction_sum + target_sum > 0,
                       2 * intersection / (prediction_sum + target_sum).clamp(min=1),
                       torch.ones_like(union))
    sensitivity = torch.where(target_sum > 0,
                              intersection / target_sum.clamp(min=1),
                              torch.ones_like(target_sum))
    return jaccard, dice, sensitivity

def mask_surface(mask):
    eroded = ndimage.binary_erosion(mask, border_value=0)
    return np.logical_xor(mask, eroded)

def hd95_single(prediction, target):
    prediction, target = prediction.astype(bool), target.astype(bool)
    height, width = target.shape
    if not prediction.any() and not target.any():
        return 0.0
    if not prediction.any() or not target.any():
        return float(math.hypot(height, width))
    prediction_surface = mask_surface(prediction)
    target_surface = mask_surface(target)
    distance_to_target = ndimage.distance_transform_edt(~target_surface)
    distance_to_prediction = ndimage.distance_transform_edt(~prediction_surface)
    distances = np.concatenate([
        distance_to_target[prediction_surface], distance_to_prediction[target_surface]
    ])
    return float(np.percentile(distances, 95)) if distances.size else 0.0

@torch.inference_mode()
def ensemble_probability(models_list, inputs):
    total = torch.zeros(
        (inputs.shape[0], 1, inputs.shape[2], inputs.shape[3]),
        device=inputs.device, dtype=torch.float32,
    )
    for model in models_list:
        model.eval()
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_AMP):
            probability = torch.sigmoid(model(inputs).float())
        if not bool(torch.isfinite(probability).all()):
            raise FloatingPointError('Model sinh NaN/Inf khi evaluation')
        total += probability
    return total / len(models_list)

@torch.inference_mode()
def evaluate_full(models_list, loader, include_hd95=True):
    sums = {'jaccard': 0.0, 'threshold_jaccard': 0.0,
            'dice': 0.0, 'sensitivity': 0.0}
    hd95_sum, sample_count = 0.0, 0
    for inputs, targets in loader:
        inputs = prepare_inputs(inputs)
        targets_device = targets.to(DEVICE, non_blocking=True)
        probability = ensemble_probability(models_list, inputs)
        jaccard, dice, sensitivity = per_image_scores(probability, targets_device)
        sums['jaccard'] += float(jaccard.sum())
        sums['threshold_jaccard'] += float((jaccard * (jaccard >= JACCARD_CUT)).sum())
        sums['dice'] += float(dice.sum())
        sums['sensitivity'] += float(sensitivity.sum())
        sample_count += targets.shape[0]
        if include_hd95:
            prediction_np = (probability > PRED_THRESHOLD).cpu().numpy()[:, 0]
            target_np = targets.numpy()[:, 0] > 0.5
            hd95_sum += sum(
                hd95_single(prediction, target)
                for prediction, target in zip(prediction_np, target_np)
            )
    if sample_count == 0:
        raise RuntimeError('Evaluation loader rỗng')
    metrics = {key: value / sample_count for key, value in sums.items()}
    if include_hd95:
        metrics['hd95'] = hd95_sum / sample_count
    return metrics

metric_target = torch.zeros(2, 1, 8, 8, device=DEVICE)
metric_target[0, :, :4] = 1
metric_target[1, :, :, :4] = 1
metric_probability = metric_target.clone()
metric_probability[1] = 0
metric_probability[1, :, :, :2] = 1
assert torch.allclose(
    per_image_dice(metric_probability, metric_target),
    torch.tensor([1.0, 2 / 3], device=DEVICE), atol=1e-6,
)
del metric_target, metric_probability
print('Loss/search-space/metric checks: OK')


In [ ]:
def atomic_torch_save(payload, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    torch.save(payload, temporary)
    temporary.replace(path)

def atomic_json_save(payload, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(
        json.dumps(payload, indent=2, ensure_ascii=False), encoding='utf-8'
    )
    temporary.replace(path)

def trusted_torch_load(path, map_location='cpu'):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)

def config_digest(config):
    canonical = json.dumps(config, sort_keys=True, separators=(',', ':'), ensure_ascii=True)
    return hashlib.sha256(canonical.encode('utf-8')).hexdigest()

def make_scaler():
    try:
        return torch.amp.GradScaler('cuda', enabled=USE_AMP)
    except TypeError:
        return torch.cuda.amp.GradScaler(enabled=USE_AMP)

def trainable_parameters(model):
    parameters = [p for p in model.parameters() if p.requires_grad]
    if not parameters:
        raise RuntimeError('Không có parameter trainable')
    return parameters

def update_global_best(model, candidate_score, candidate_label,
                       best_score=-float('inf'), best_label=None, best_state=None):
    candidate_score = float(candidate_score)
    if not np.isfinite(candidate_score):
        raise FloatingPointError(f'Best-Dice candidate không hợp lệ: {candidate_label}')
    if best_state is None:
        best_state = cpu_state_dict(model)
        return candidate_score, str(candidate_label), best_state
    if candidate_score > float(best_score):
        refresh_cpu_state_dict(model, best_state)
        return candidate_score, str(candidate_label), best_state
    # Stage mới kém hơn: khôi phục global Best Dice trước khi train stage kế tiếp.
    load_model_state(model, best_state)
    return float(best_score), best_label, best_state

def build_optimizer(model, learning_rate, optimizer_config):
    parameters = trainable_parameters(model)
    name = optimizer_config['name']
    if name == 'Adam':
        return torch.optim.Adam(
            parameters, lr=learning_rate,
            weight_decay=float(optimizer_config.get('weight_decay', 0.0)),
        )
    if name == 'AdamW':
        return torch.optim.AdamW(
            parameters, lr=learning_rate,
            weight_decay=float(optimizer_config['weight_decay']),
        )
    if name == 'SGD':
        return torch.optim.SGD(
            parameters, lr=learning_rate,
            momentum=float(optimizer_config['momentum']),
            weight_decay=float(optimizer_config['weight_decay']),
        )
    raise ValueError(f"Optimizer không hỗ trợ: {name}")

def train_batch(model, optimizer, scaler, loss_fn, inputs, targets):
    inputs = prepare_inputs(inputs)
    targets = targets.to(DEVICE, non_blocking=True)
    optimizer.zero_grad(set_to_none=True)
    with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_AMP):
        loss = loss_fn(model(inputs).float(), targets)
    if not bool(torch.isfinite(loss)):
        raise FloatingPointError('Train loss là NaN/Inf')
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    nn.utils.clip_grad_norm_(
        (p for group in optimizer.param_groups for p in group['params']),
        GRAD_CLIP_NORM,
    )
    scaler.step(optimizer)
    scaler.update()
    return loss.detach()

def lr_find(model, records, size, batch_size, loss_fn, optimizer_config,
            stage_seed, freeze_bn):
    # Hàm chỉ nhận train records; không có đối số validation để tránh leakage.
    set_seed(stage_seed)
    original_state = cpu_state_dict(model)
    optimizer = build_optimizer(model, LR_MIN, optimizer_config)
    scaler = make_scaler()
    loader = make_loader(
        records, size, train=True, batch_size=batch_size,
        seed=stage_seed, drop_last=False,
    )
    iterator = iter(loader)
    multiplier = (LR_MAX / LR_MIN) ** (1 / max(1, LRF_ITERS - 1))
    learning_rates, losses = [], []
    moving_average, best_loss = 0.0, float('inf')
    for iteration in range(LRF_ITERS):
        try:
            inputs, targets = next(iterator)
        except StopIteration:
            iterator = iter(loader)
            inputs, targets = next(iterator)
        learning_rate = LR_MIN * multiplier ** iteration
        for group in optimizer.param_groups:
            group['lr'] = learning_rate
        set_train_mode(model, freeze_bn)
        try:
            raw_loss = float(
                train_batch(model, optimizer, scaler, loss_fn, inputs, targets).cpu()
            )
        except FloatingPointError:
            break
        moving_average = LRF_BETA * moving_average + (1 - LRF_BETA) * raw_loss
        smoothed = moving_average / (1 - LRF_BETA ** (iteration + 1))
        if not np.isfinite(smoothed) or (iteration > 5 and smoothed > 4 * best_loss):
            break
        best_loss = min(best_loss, smoothed)
        learning_rates.append(learning_rate)
        losses.append(smoothed)

    del iterator, loader, optimizer, scaler, inputs, targets
    load_model_state(model, original_state)
    del original_state
    gc.collect()
    set_seed(stage_seed)

    learning_rates = np.asarray(learning_rates, dtype=float)
    losses = np.asarray(losses, dtype=float)
    if len(learning_rates) < 3:
        raise FloatingPointError('LR Finder dừng quá sớm; không đủ điểm hợp lệ')
    gradient = np.gradient(losses, np.log10(learning_rates))
    steepest_lr = float(learning_rates[int(np.argmin(gradient))])
    min_loss_lr = float(learning_rates[int(np.argmin(losses))])
    optimizer_name = optimizer_config['name']
    cap = (MAX_LR_UNFROZEN if freeze_bn else MAX_LR_FROZEN)[optimizer_name]
    selected_lr = max(
        LR_MIN,
        min(steepest_lr * LR_SELECTION_FACTOR, min_loss_lr / 10, cap),
    )
    print(
        f'LR Finder {size}px {optimizer_name}: selected={selected_lr:.2e} '
        f'(steepest={steepest_lr:.2e}, min-loss={min_loss_lr:.2e}, cap={cap:.2e})'
    )
    return selected_lr, learning_rates.tolist(), losses.tolist()

def stlr_lambda(total_steps, cut_frac=CUT_FRAC, ratio=STLR_RATIO):
    cut = max(1, int(total_steps * cut_frac))
    def schedule(step):
        if step < cut:
            progress = step / cut
        else:
            denominator = cut * (1 / cut_frac - 1)
            progress = max(0.0, 1 - (step - cut) / max(1, denominator))
        return (1 + progress * (ratio - 1)) / ratio
    return schedule

def train_one_epoch(model, loader, optimizer, scaler, scheduler, loss_fn, freeze_bn):
    set_train_mode(model, freeze_bn)
    loss_sum = 0.0
    sample_count = 0
    for inputs, targets in loader:
        loss = train_batch(model, optimizer, scaler, loss_fn, inputs, targets)
        batch_count = inputs.shape[0]
        loss_sum += float(loss.cpu()) * batch_count
        sample_count += batch_count
        scheduler.step()
    if sample_count == 0:
        raise RuntimeError('Train loader rỗng')
    return loss_sum / sample_count, sample_count

def fit_stage_optuna(model, train_subset, validation_subset, size, batch_size,
                     epochs, freeze_bn, loss_fn, optimizer_config, stage_seed,
                     trial, step_offset, tag, prior_best_score=None):
    selected_lr, lrf_lrs, lrf_losses = lr_find(
        model, train_subset, size, batch_size, loss_fn, optimizer_config,
        stage_seed, freeze_bn,
    )
    validation_loader = make_loader(
        validation_subset, size, train=False, batch_size=batch_size,
        seed=stage_seed,
    )
    steps_per_epoch = math.ceil(len(train_subset) / batch_size)
    optimizer = build_optimizer(model, selected_lr, optimizer_config)
    scaler = make_scaler()
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, stlr_lambda(epochs * steps_per_epoch)
    )
    best_score, best_epoch, best_state = -float('inf'), None, None
    history = []
    for epoch in range(epochs):
        epoch_started = time.perf_counter()
        train_loader = make_loader(
            train_subset, size, train=True, batch_size=batch_size,
            seed=stage_seed + epoch, drop_last=False,
        )
        train_loss, samples_seen = train_one_epoch(
            model, train_loader, optimizer, scaler, scheduler, loss_fn, freeze_bn
        )
        del train_loader
        validation_dice = evaluate_dice(model, validation_loader)
        if not np.isfinite(validation_dice):
            raise FloatingPointError('Internal-validation Dice là NaN/Inf')
        row = {
            'epoch': epoch + 1, 'train_loss': train_loss,
            'validation_dice': validation_dice,
            'lr': optimizer.param_groups[0]['lr'],
        }
        history.append(row)
        if validation_dice > best_score:
            best_score, best_epoch = validation_dice, epoch + 1
            if best_state is None:
                best_state = cpu_state_dict(model)
            else:
                refresh_cpu_state_dict(model, best_state)
        report_step = step_offset + epoch
        reported_best = best_score if prior_best_score is None else max(
            float(prior_best_score), float(best_score)
        )
        trial.report(reported_best, step=report_step)
        seconds = time.perf_counter() - epoch_started
        print(
            f'[{tag}] epoch {epoch + 1:02d}/{epochs} loss={train_loss:.4f} '
            f'dice={validation_dice:.4f} stage_best={best_score:.4f} '
            f'global_best={reported_best:.4f} '
            f'time={seconds:.1f}s img/s={samples_seen / seconds:.1f}'
        )
        if trial.should_prune():
            trial.set_user_attr('pruned_at', f'{tag}:epoch={epoch + 1}')
            raise optuna.TrialPruned(f'Pruned at {tag} epoch {epoch + 1}')

    if best_state is None:
        raise RuntimeError(f'{tag}: không tạo được best state')
    load_model_state(model, best_state)
    del best_state, validation_loader, optimizer, scaler, scheduler
    gc.collect()
    return {
        'best_dice': float(best_score), 'best_epoch': int(best_epoch),
        'selected_lr': float(selected_lr), 'history': history,
        'lrf_lrs': lrf_lrs, 'lrf_losses': lrf_losses,
    }

def fit_stage_resumable(model, train_subset, validation_subset, size, batch_size,
                        epochs, freeze_bn, loss_fn, optimizer_config, stage_seed,
                        stage_dir, contract, tag):
    stage_dir = Path(stage_dir)
    stage_dir.mkdir(parents=True, exist_ok=True)
    progress_path = stage_dir / 'progress.pt'
    best_path = stage_dir / 'best.pt'
    saved = None
    if progress_path.is_file():
        saved = trusted_torch_load(progress_path, map_location='cpu')
        if saved.get('contract') != contract:
            raise RuntimeError(f'Checkpoint không khớp contract: {progress_path}')
        if saved.get('complete'):
            best_payload = trusted_torch_load(best_path, map_location='cpu')
            if best_payload.get('contract') != contract:
                raise RuntimeError(f'Best checkpoint không khớp: {best_path}')
            load_model_state(model, best_payload['model_state'])
            print(f'[resume complete] {tag}')
            return saved['result']

    set_seed(stage_seed)
    validation_loader = make_loader(
        validation_subset, size, train=False, batch_size=batch_size,
        seed=stage_seed,
    )
    steps_per_epoch = math.ceil(len(train_subset) / batch_size)

    if saved is None:
        selected_lr, lrf_lrs, lrf_losses = lr_find(
            model, train_subset, size, batch_size, loss_fn, optimizer_config,
            stage_seed, freeze_bn,
        )
        optimizer = build_optimizer(model, selected_lr, optimizer_config)
        scaler = make_scaler()
        scheduler = torch.optim.lr_scheduler.LambdaLR(
            optimizer, stlr_lambda(epochs * steps_per_epoch)
        )
        best_score, best_epoch, best_state = -float('inf'), None, None
        history, start_epoch = [], 0
    else:
        selected_lr = float(saved['selected_lr'])
        lrf_lrs, lrf_losses = saved['lrf_lrs'], saved['lrf_losses']
        optimizer = build_optimizer(model, selected_lr, optimizer_config)
        scaler = make_scaler()
        scheduler = torch.optim.lr_scheduler.LambdaLR(
            optimizer, stlr_lambda(epochs * steps_per_epoch)
        )
        load_model_state(model, saved['model_state'])
        optimizer.load_state_dict(saved['optimizer_state'])
        scheduler.load_state_dict(saved['scheduler_state'])
        scaler.load_state_dict(saved['scaler_state'])
        best_payload = trusted_torch_load(best_path, map_location='cpu')
        if best_payload.get('contract') != contract:
            raise RuntimeError(f'Best checkpoint không khớp: {best_path}')
        best_state = best_payload['model_state']
        best_score = float(best_payload['best_score'])
        best_epoch = int(best_payload['best_epoch'])
        history = list(saved['history'])
        start_epoch = int(saved['next_epoch'])
        print(f'[resume] {tag}: bắt đầu epoch {start_epoch + 1}')
        saved = None
        gc.collect()

    for epoch in range(start_epoch, epochs):
        epoch_started = time.perf_counter()
        train_loader = make_loader(
            train_subset, size, train=True, batch_size=batch_size,
            seed=stage_seed + epoch, drop_last=False,
        )
        train_loss, samples_seen = train_one_epoch(
            model, train_loader, optimizer, scaler, scheduler, loss_fn, freeze_bn
        )
        del train_loader
        validation_dice = evaluate_dice(model, validation_loader)
        if not np.isfinite(validation_dice):
            raise FloatingPointError('Validation Dice là NaN/Inf')
        history.append({
            'epoch': epoch + 1, 'train_loss': train_loss,
            'validation_dice': validation_dice,
            'lr': optimizer.param_groups[0]['lr'],
        })
        if validation_dice > best_score:
            best_score, best_epoch = validation_dice, epoch + 1
            if best_state is None:
                best_state = cpu_state_dict(model)
            else:
                refresh_cpu_state_dict(model, best_state)

        should_save = (epoch + 1) % SAVE_LAST_EVERY == 0 or epoch + 1 == epochs
        if should_save:
            atomic_torch_save({
                'model_state': best_state,
                'best_score': float(best_score),
                'best_epoch': int(best_epoch),
                'contract': contract,
            }, best_path)
            atomic_torch_save({
                'complete': False, 'contract': contract,
                'next_epoch': epoch + 1,
                'selected_lr': float(selected_lr),
                'lrf_lrs': lrf_lrs, 'lrf_losses': lrf_losses,
                'history': history,
                'model_state': cpu_state_dict(model),
                'optimizer_state': optimizer.state_dict(),
                'scheduler_state': scheduler.state_dict(),
                'scaler_state': scaler.state_dict(),
            }, progress_path)
        seconds = time.perf_counter() - epoch_started
        print(
            f'[{tag}] epoch {epoch + 1:02d}/{epochs} loss={train_loss:.4f} '
            f'dice={validation_dice:.4f} best={best_score:.4f} '
            f'time={seconds:.1f}s img/s={samples_seen / seconds:.1f} {memory_status()}'
        )

    if best_state is None:
        raise RuntimeError(f'{tag}: không tạo được best state')
    load_model_state(model, best_state)
    result = {
        'best_dice': float(best_score), 'best_epoch': int(best_epoch),
        'selected_lr': float(selected_lr), 'history': history,
        'lrf_lrs': lrf_lrs, 'lrf_losses': lrf_losses,
        'best_path': str(best_path),
    }
    atomic_torch_save({
        'model_state': best_state, 'best_score': float(best_score),
        'best_epoch': int(best_epoch), 'contract': contract,
    }, best_path)
    atomic_torch_save({
        'complete': True, 'contract': contract, 'result': result,
    }, progress_path)
    del best_state, validation_loader, optimizer, scaler, scheduler
    gc.collect()
    return result


In [ ]:
RUN_CONFIG = {
    'protocol_version': PROTOCOL_VERSION,
    'dataset_handle': DATASET_HANDLE,
    'expected_counts': EXPECTED_COUNTS,
    'seed': SEED,
    'hpo_split': {
        'val_fraction': HPO_VAL_FRACTION,
        'train_ids_sha256': ids_digest(hpo_train_records),
        'validation_ids_sha256': ids_digest(hpo_val_records),
    },
    'search_space': {
        'batch_size_128': list(SEARCH_BATCHES_128),
        'batch_size_256_cap': MAX_BATCH_256,
        'optimizers': {
            'choices': ['SGD', 'Adam', 'AdamW'],
            'SGD_momentum': [0.80, 0.99],
            'SGD_weight_decay_log': [1e-6, 1e-3],
            'Adam_weight_decay': 0.0,
            'AdamW_weight_decay_log': [1e-6, 1e-3],
        },
        'losses': {
            'choices': ['BCE', 'Dice', 'Combo', 'Tversky', 'Focal'],
            'Combo_or_Tversky_alpha': [0.2, 0.8],
            'Focal_gamma': [1.0, 4.0],
        },
    },
    'sampler': {
        'name': 'TPESampler', 'n_startup_trials': 10,
        'rng_resume': 'atomic pickle; fallback seed=base_seed+existing_trial_count',
    },
    'top_k': TOP_K,
    'hpo_epochs': [HPO_FROZEN_EPOCHS, HPO_UNFROZEN_EPOCHS],
    'confirmation_epochs': [CONFIRM_FROZEN_EPOCHS, CONFIRM_UNFROZEN_EPOCHS],
    'final': {
        'folds': N_FINAL_FOLDS,
        'epochs': [FINAL_FROZEN_EPOCHS, FINAL_UNFROZEN_EPOCHS],
        'sizes': [128, 256],
    },
    'lr_finder': {
        'iterations': LRF_ITERS, 'min': LR_MIN, 'max': LR_MAX,
        'beta': LRF_BETA, 'selection_factor': LR_SELECTION_FACTOR,
        'max_lr_frozen': MAX_LR_FROZEN,
        'max_lr_unfrozen': MAX_LR_UNFROZEN,
    },
    'scheduler': {'name': 'STLR', 'cut_frac': CUT_FRAC, 'ratio': STLR_RATIO},
    'pruner': {
        'name': 'PercentilePruner', 'percentile': 75.0,
        'n_startup_trials': 10, 'n_warmup_steps': 10,
        'interval_steps': 1, 'n_min_trials': 10,
    },
    'augmentation': 'D4+Affine(rotate44,scale1.0-1.05)+BrightnessContrast0.2',
    'augmentation_only_train': True,
    'selection_metric': 'per-image Dice @ threshold 0.5',
    'preprocessing': 'opencv_bgr+upstream_simplest_cb_1pct+bgr_to_rgb+resize+imagenet_normalize_on_cuda0',
    'color_balance': {
        'method': 'simplest_cb', 'percent': COLOR_BALANCE_PERCENT,
        'tail_percent_per_side': COLOR_BALANCE_PERCENT / 2,
        'source': COLOR_BALANCE_SOURCE,
        'decode': 'cv2.imread BGR', 'postprocess': 'cv2 BGR-to-RGB',
        'applied_before_resize': True,
    },
    'pretrained': PRETRAINED,
    'amp': USE_AMP,
    'fp32_transposed_conv': FP32_TRANSPOSED_CONV,
    'grad_clip_norm': GRAD_CLIP_NORM,
    'prediction_threshold': PRED_THRESHOLD,
    'jaccard_cut': JACCARD_CUT,
    'cuda_visible_devices': '0',
    'channels_last': USE_CHANNELS_LAST,
    'optuna_version': optuna.__version__,
    'num_workers': NUM_WORKERS,
    'cache_resized_train_uint8': CACHE_RESIZED_TRAIN_IN_RAM,
    'fast_dev_run': FAST_DEV_RUN,
}
RUN_CONFIG_SHA256 = config_digest(RUN_CONFIG)
config_path = RUN_DIR / 'config.json'
if config_path.is_file():
    previous_config = json.loads(config_path.read_text(encoding='utf-8'))
    if config_digest(previous_config) != RUN_CONFIG_SHA256:
        raise RuntimeError(
            'RUN_DIR chứa artifact của config khác. Đổi RUN_NAME hoặc xóa/đổi output cũ.'
        )
else:
    atomic_json_save(RUN_CONFIG, config_path)
atomic_json_save({
    'target_total_trials': TARGET_TOTAL_TRIALS,
    'optuna_timeout_seconds': OPTUNA_TIMEOUT_SECONDS,
    'run_hpo': RUN_HPO,
    'run_confirm_256': RUN_CONFIRM_256,
    'run_final_cv': RUN_FINAL_CV,
    'run_official_evaluation': RUN_OFFICIAL_EVALUATION,
}, RUN_DIR / 'execution_settings.json')

def preflight_batch(size, batch_size):
    model = to_runtime(UNet34(pretrained=False))
    optimizer = build_optimizer(
        model, 1e-4, {'name': 'Adam', 'weight_decay': 0.0}
    )
    scaler = make_scaler()
    loss_fn = nn.BCEWithLogitsLoss()
    inputs = torch.randint(0, 256, (batch_size, 3, size, size), dtype=torch.uint8)
    targets = torch.randint(0, 2, (batch_size, 1, size, size), dtype=torch.float32)
    try:
        loss = float(train_batch(
            model, optimizer, scaler, loss_fn, inputs, targets
        ).cpu())
        torch.cuda.synchronize()
    except torch.cuda.OutOfMemoryError as error:
        raise RuntimeError(
            f'Batch {batch_size} không vừa cuda:0 ở {size}px. Giảm batch config.'
        ) from error
    finally:
        del model, optimizer, scaler, loss_fn, inputs, targets
        gc.collect()
        torch.cuda.empty_cache()
    print(f'Preflight {size}px batch={batch_size}: loss={loss:.4f}')

preflight_batch(128, max(SEARCH_BATCHES_128))
if RUN_CONFIRM_256 or RUN_FINAL_CV or RUN_OFFICIAL_EVALUATION:
    preflight_batch(256, MAX_BATCH_256)
print('Protocol SHA256:', RUN_CONFIG_SHA256)


## 1. Optuna HPO ở 128 px

Mọi trial dùng cùng một split và cùng seed để giảm nhiễu khi so cấu hình. LR Finder chỉ nhìn batch có augmentation từ HPO train subset. Internal-validation loader luôn sạch và chỉ dùng để report/prune/chọn Best Dice. Pruner được gọi sau **mỗi epoch**, không đợi hết 30 epoch frozen như pipeline cũ.

SQLite `optuna.db`, bảng trial CSV và Top-K checkpoint được ghi ngay trong `/kaggle/working`; `n_jobs=1` bảo đảm notebook chỉ chạy tuần tự trên `cuda:0`.

`PRUNER_PERCENTILE=25` với bài toán maximize là cấu hình mạnh: tại các step đủ dữ liệu, pruner giữ xấp xỉ nhóm top 25%. Muốn chỉ loại khoảng bottom 25% thì đặt 75 trước lần chạy đầu.


In [ ]:
class TopKManager:
    def __init__(self, directory, k, contract_sha256):
        self.directory = Path(directory)
        self.directory.mkdir(parents=True, exist_ok=True)
        self.manifest_path = self.directory / 'manifest.json'
        self.k = int(k)
        self.contract_sha256 = str(contract_sha256)
        if self.manifest_path.is_file():
            payload = json.loads(self.manifest_path.read_text(encoding='utf-8'))
            if payload.get('contract_sha256') != self.contract_sha256:
                raise RuntimeError('Top-K manifest thuộc protocol/config khác')

    def entries(self):
        if not self.manifest_path.is_file():
            return []
        payload = json.loads(self.manifest_path.read_text(encoding='utf-8'))
        entries = []
        for entry in payload.get('entries', []):
            if (self.directory / entry['checkpoint']).is_file():
                entries.append(entry)
        return sorted(entries, key=lambda row: float(row['score']), reverse=True)

    def _write(self, entries):
        atomic_json_save({
            'contract_sha256': self.contract_sha256,
            'selection_metric': 'internal_validation_dice',
            'entries': entries,
        }, self.manifest_path)

    def update(self, trial_number, score, model, params, metadata):
        entries = [
            row for row in self.entries()
            if int(row['trial_number']) != int(trial_number)
        ]
        filename = f'trial_{int(trial_number):04d}.pt'
        checkpoint_path = self.directory / filename
        candidate = {
            'trial_number': int(trial_number),
            'score': float(score),
            'checkpoint': filename,
            'params': params,
            'metadata': metadata,
        }
        entries.append(candidate)
        entries.sort(key=lambda row: float(row['score']), reverse=True)
        keep, removed = entries[:self.k], entries[self.k:]
        if any(int(row['trial_number']) == int(trial_number) for row in keep):
            atomic_torch_save({
                'model_state': cpu_state_dict(model),
                'trial_number': int(trial_number),
                'score': float(score),
                'params': params,
                'metadata': metadata,
                'contract_sha256': self.contract_sha256,
            }, checkpoint_path)
        for row in removed:
            path = self.directory / row['checkpoint']
            if path.is_file():
                path.unlink()
        self._write(keep)

    def reconcile(self, study):
        complete_numbers = {
            trial.number for trial in study.get_trials(deepcopy=False)
            if trial.state == optuna.trial.TrialState.COMPLETE
        }
        valid, removed = [], []
        for entry in self.entries():
            if int(entry['trial_number']) in complete_numbers:
                valid.append(entry)
            else:
                removed.append(entry)
        for entry in removed:
            path = self.directory / entry['checkpoint']
            if path.is_file():
                path.unlink()
        self._write(valid[:self.k])

topk_manager = TopKManager(TOPK_DIR, TOP_K, RUN_CONFIG_SHA256)

def create_or_load_study():
    storage_url = f"sqlite:///{STUDY_DB.resolve().as_posix()}"
    study_name = f'unet34_hus_best_dice_{RUN_NAME}'
    # Optuna RDB không lưu RNG state của sampler. Ưu tiên khôi phục sampler pickle
    # theo đúng recipe chính thức; nếu file thiếu thì dịch seed theo số trial cũ.
    existing_trial_count = 0
    if STUDY_DB.is_file():
        try:
            previous = optuna.load_study(study_name=study_name, storage=storage_url)
            existing_trial_count = len(previous.trials)
            del previous
        except (KeyError, RuntimeError):
            existing_trial_count = 0
    sampler = None
    if SAMPLER_PATH.is_file():
        try:
            with SAMPLER_PATH.open('rb') as stream:
                sampler = pickle.load(stream)
            sampler_resume_mode = 'restored_pickle'
            session_sampler_seed = None
        except (pickle.UnpicklingError, EOFError, AttributeError,
                ModuleNotFoundError, ImportError) as error:
            print(f'Không restore được sampler pickle ({error}); dùng fallback seed.')
    if sampler is None:
        session_sampler_seed = SEED + existing_trial_count
        sampler = optuna.samplers.TPESampler(
            seed=session_sampler_seed, n_startup_trials=10
        )
        sampler_resume_mode = 'new_or_fallback_seed'
    pruner = optuna.pruners.PercentilePruner(
        percentile=75.0,
        n_startup_trials=10,
        n_warmup_steps=10,
        interval_steps=1,
        n_min_trials=10,
    )
    study = optuna.create_study(
        study_name=study_name,
        storage=storage_url,
        direction='maximize',
        sampler=sampler,
        pruner=pruner,
        load_if_exists=True,
    )
    saved_hash = study.user_attrs.get('run_config_sha256')
    if saved_hash is None:
        study.set_user_attr('run_config_sha256', RUN_CONFIG_SHA256)
    elif saved_hash != RUN_CONFIG_SHA256:
        raise RuntimeError('Optuna DB thuộc protocol/config khác')
    study.set_user_attr('dataset_handle', DATASET_HANDLE)
    study.set_user_attr('selection_metric', 'internal_validation_dice')
    study.set_user_attr('latest_sampler_resume_mode', sampler_resume_mode)
    study.set_user_attr('latest_session_sampler_seed', session_sampler_seed)
    print({
        'sampler_resume_mode': sampler_resume_mode,
        'session_sampler_seed': session_sampler_seed,
        'existing_trials': existing_trial_count,
    })
    return study

def persist_sampler(study, frozen_trial=None):
    del frozen_trial
    temporary = SAMPLER_PATH.with_suffix(SAMPLER_PATH.suffix + '.tmp')
    with temporary.open('wb') as stream:
        pickle.dump(study.sampler, stream)
    temporary.replace(SAMPLER_PATH)

def objective(trial):
    model = None
    try:
        # Cùng seed/model init/augmentation stream cho mọi trial để so sánh công bằng.
        set_seed(SEED)
        sampled = sample_configuration(trial)
        batch_size = batch_for_size(sampled['batch_size'], 128)
        loss_fn = build_loss(sampled['loss'])
        model = to_runtime(UNet34(pretrained=PRETRAINED))

        freeze_first_layer_group(model)
        frozen = fit_stage_optuna(
            model, hpo_train_records, hpo_val_records,
            size=128, batch_size=batch_size,
            epochs=HPO_FROZEN_EPOCHS, freeze_bn=False,
            loss_fn=loss_fn, optimizer_config=sampled['optimizer'],
            stage_seed=SEED, trial=trial, step_offset=0,
            tag=f'trial {trial.number} | 128px | frozen',
        )
        global_best_score, global_best_stage, global_best_state = update_global_best(
            model, frozen['best_dice'], '128px_frozen'
        )
        unfreeze_with_frozen_bn(model)
        unfrozen = fit_stage_optuna(
            model, hpo_train_records, hpo_val_records,
            size=128, batch_size=batch_size,
            epochs=HPO_UNFROZEN_EPOCHS, freeze_bn=True,
            loss_fn=loss_fn, optimizer_config=sampled['optimizer'],
            stage_seed=SEED + 500, trial=trial,
            step_offset=HPO_FROZEN_EPOCHS,
            tag=f'trial {trial.number} | 128px | unfrozen',
            prior_best_score=global_best_score,
        )
        global_best_score, global_best_stage, global_best_state = update_global_best(
            model, unfrozen['best_dice'], '128px_unfrozen',
            global_best_score, global_best_stage, global_best_state,
        )
        score = float(global_best_score)
        metadata = {
            'sampled': sampled,
            'frozen_best_dice': float(frozen['best_dice']),
            'frozen_best_epoch': int(frozen['best_epoch']),
            'frozen_lr': float(frozen['selected_lr']),
            'unfrozen_best_dice': float(unfrozen['best_dice']),
            'unfrozen_best_epoch': int(unfrozen['best_epoch']),
            'unfrozen_lr': float(unfrozen['selected_lr']),
            'global_best_dice': score,
            'selected_stage': global_best_stage,
            'train_ids_sha256': ids_digest(hpo_train_records),
            'validation_ids_sha256': ids_digest(hpo_val_records),
        }
        for key, value in metadata.items():
            if key != 'sampled':
                trial.set_user_attr(key, value)
        trial.set_user_attr('sampled_json', json.dumps(sampled, sort_keys=True))
        topk_manager.update(
            trial.number, score, model, dict(trial.params), metadata
        )
        del global_best_state
        return score
    except torch.cuda.OutOfMemoryError as error:
        trial.set_user_attr('failure_reason', f'CUDA OOM: {error}')
        raise optuna.TrialPruned('CUDA OOM; trial được prune') from error
    finally:
        if model is not None:
            del model
        gc.collect()
        torch.cuda.empty_cache()

study = create_or_load_study()
persist_sampler(study)
topk_manager.reconcile(study)
terminal_states = {
    optuna.trial.TrialState.COMPLETE,
    optuna.trial.TrialState.PRUNED,
    optuna.trial.TrialState.FAIL,
}

def terminal_trial_count(study):
    return sum(
        trial.state in terminal_states
        for trial in study.get_trials(deepcopy=False)
    )

before_count = terminal_trial_count(study)
before_topk_count = len(topk_manager.entries())
needed_for_target = max(0, TARGET_TOTAL_TRIALS - before_count)
needed_for_topk = max(0, TOP_K - before_topk_count)
remaining_trials = max(needed_for_target, needed_for_topk)
print(
    f'Optuna terminal trials hiện có={before_count}, target={TARGET_TOTAL_TRIALS}, '
    f'Top-K={before_topk_count}/{TOP_K}, cần chạy thêm ít nhất={remaining_trials}'
)
if RUN_HPO and remaining_trials > 0:
    # Không catch RuntimeError/AcceleratorError: CUDA illegal access làm context hỏng,
    # cần restart session thay vì giả vờ tiếp tục. Chỉ FloatingPointError thành FAIL trial.
    study.optimize(
        objective,
        n_trials=remaining_trials,
        timeout=OPTUNA_TIMEOUT_SECONDS,
        n_jobs=1,
        gc_after_trial=True,
        catch=(FloatingPointError,),
        callbacks=[persist_sampler],
        show_progress_bar=False,
    )

topk_manager.reconcile(study)
trials_frame = study.trials_dataframe(attrs=('number', 'value', 'params', 'state', 'user_attrs'))
trials_frame.to_csv(RUN_DIR / 'trials.csv', index=False)

terminal_count = terminal_trial_count(study)
complete_trials = [
    trial for trial in study.get_trials(deepcopy=False)
    if trial.state == optuna.trial.TrialState.COMPLETE
]
top_entries = topk_manager.entries()
HPO_READY = terminal_count >= TARGET_TOTAL_TRIALS and len(top_entries) >= TOP_K
state_counts = pd.Series([str(t.state.name) for t in study.trials]).value_counts().to_dict()
print('Study states:', state_counts)
print(f'HPO_READY={HPO_READY} | Top-K checkpoints={len(top_entries)}/{TOP_K}')
if complete_trials:
    print('Best trial:', study.best_trial.number, 'Dice=', study.best_value)
    display(
        trials_frame.sort_values('value', ascending=False, na_position='last').head(10)
    )
if not HPO_READY:
    print(
        'HPO chưa đủ target/Top-K. Hãy Save Version, attach output ở session sau, '
        'đặt RESTORE_FROM rồi chạy lại. Confirmation sẽ không được phép chạy sớm.'
    )


## 2. Xác nhận Top-K ở 256 px và tự khóa winner

Mỗi checkpoint Top-K kế thừa trực tiếp trọng số tốt nhất ở 128 px, giữ nguyên optimizer/loss của trial và train tiếp ở 256 px. Batch 256 được chặn tối đa 8 để vừa một GPU T4. Đây vẫn là giai đoạn tuning trên **cùng internal validation subset**, không phải báo cáo độc lập; official validation/test vẫn chưa được dùng.


In [ ]:
def confirmation_contract(entry, sampled):
    return {
        'protocol_sha256': RUN_CONFIG_SHA256,
        'phase': 'confirmation_256',
        'trial_number': int(entry['trial_number']),
        'params': entry['params'],
        'sampled': sampled,
        'train_ids_sha256': ids_digest(hpo_train_records),
        'validation_ids_sha256': ids_digest(hpo_val_records),
    }

def confirm_candidate(entry):
    trial_number = int(entry['trial_number'])
    sampled = configuration_from_params(entry['params'])
    candidate_dir = CONFIRM_DIR / f'trial_{trial_number:04d}'
    final_path = candidate_dir / 'confirmed_best.pt'
    contract = confirmation_contract(entry, sampled)
    contract_sha = config_digest(contract)
    if final_path.is_file():
        payload = trusted_torch_load(final_path, map_location='cpu')
        if payload.get('contract_sha256') != contract_sha:
            raise RuntimeError(f'Confirmation checkpoint sai contract: {final_path}')
        print(f'[resume complete] confirmation trial {trial_number}')
        return payload['metadata']

    hpo_payload = trusted_torch_load(
        TOPK_DIR / entry['checkpoint'], map_location='cpu'
    )
    if hpo_payload.get('contract_sha256') != RUN_CONFIG_SHA256:
        raise RuntimeError(f'HPO checkpoint sai protocol: trial {trial_number}')
    model = to_runtime(UNet34(pretrained=False))
    load_model_state(model, hpo_payload['model_state'])
    del hpo_payload
    loss_fn = build_loss(sampled['loss'])
    batch_size = batch_for_size(sampled['batch_size'], 256)
    # Mọi candidate dùng cùng augmentation/order seed để comparison công bằng.
    stage_seed = SEED + 20_000
    validation_loader = make_loader(
        hpo_val_records, 256, train=False, batch_size=batch_size, seed=SEED
    )
    initial_dice = evaluate_dice(model, validation_loader)
    global_best_score, global_best_stage, global_best_state = update_global_best(
        model, initial_dice, 'hpo_128_checkpoint_at_256'
    )

    freeze_first_layer_group(model)
    frozen_contract = {
        **contract, 'stage': 'frozen', 'epochs': CONFIRM_FROZEN_EPOCHS,
        'batch_size': batch_size, 'stage_seed': stage_seed,
    }
    frozen = fit_stage_resumable(
        model, hpo_train_records, hpo_val_records,
        size=256, batch_size=batch_size,
        epochs=CONFIRM_FROZEN_EPOCHS, freeze_bn=False,
        loss_fn=loss_fn, optimizer_config=sampled['optimizer'],
        stage_seed=stage_seed,
        stage_dir=candidate_dir / '256px_frozen',
        contract=frozen_contract,
        tag=f'confirm trial {trial_number} | 256px | frozen',
    )
    global_best_score, global_best_stage, global_best_state = update_global_best(
        model, frozen['best_dice'], '256px_frozen',
        global_best_score, global_best_stage, global_best_state,
    )

    unfreeze_with_frozen_bn(model)
    unfrozen_contract = {
        **contract, 'stage': 'unfrozen', 'epochs': CONFIRM_UNFROZEN_EPOCHS,
        'batch_size': batch_size, 'stage_seed': stage_seed + 5,
    }
    unfrozen = fit_stage_resumable(
        model, hpo_train_records, hpo_val_records,
        size=256, batch_size=batch_size,
        epochs=CONFIRM_UNFROZEN_EPOCHS, freeze_bn=True,
        loss_fn=loss_fn, optimizer_config=sampled['optimizer'],
        stage_seed=stage_seed + 5,
        stage_dir=candidate_dir / '256px_unfrozen',
        contract=unfrozen_contract,
        tag=f'confirm trial {trial_number} | 256px | unfrozen',
    )
    global_best_score, global_best_stage, global_best_state = update_global_best(
        model, unfrozen['best_dice'], '256px_unfrozen',
        global_best_score, global_best_stage, global_best_state,
    )
    score = float(global_best_score)
    metadata = {
        'trial_number': trial_number,
        'hpo_128_dice': float(entry['score']),
        'confirmation_dice': float(score),
        'initial_checkpoint_dice_256': float(initial_dice),
        'selected_stage': global_best_stage,
        'batch_size_128': int(sampled['batch_size']),
        'batch_size_256': int(batch_size),
        'params': entry['params'],
        'sampled': sampled,
        'frozen': frozen,
        'unfrozen': unfrozen,
        'checkpoint': str(final_path),
    }
    atomic_torch_save({
        'model_state': cpu_state_dict(model),
        'metadata': metadata,
        'contract_sha256': contract_sha,
    }, final_path)
    del global_best_state, model, validation_loader
    gc.collect()
    torch.cuda.empty_cache()
    return metadata

confirmation_results_path = CONFIRM_DIR / 'confirmation_results.csv'
winner_path = RUN_DIR / 'winner.json'

if RUN_CONFIRM_256 and HPO_READY:
    confirmation_rows = []
    for rank, entry in enumerate(top_entries[:TOP_K], start=1):
        print(
            f"\n===== CONFIRM {rank}/{TOP_K}: trial={entry['trial_number']} "
            f"HPO Dice={entry['score']:.4f} ====="
        )
        confirmation_rows.append(confirm_candidate(entry))
    confirmation_rows.sort(
        key=lambda row: float(row['confirmation_dice']), reverse=True
    )
    confirmation_table = pd.DataFrame([
        {
            'rank': rank,
            'trial_number': row['trial_number'],
            'hpo_128_dice': row['hpo_128_dice'],
            'confirmation_256_dice': row['confirmation_dice'],
            'selected_stage': row['selected_stage'],
            'batch_size_128': row['batch_size_128'],
            'batch_size_256': row['batch_size_256'],
            'optimizer': row['sampled']['optimizer']['name'],
            'loss': row['sampled']['loss']['name'],
            'params_json': json.dumps(row['params'], sort_keys=True),
            'checkpoint': row['checkpoint'],
        }
        for rank, row in enumerate(confirmation_rows, start=1)
    ])
    confirmation_table.to_csv(confirmation_results_path, index=False)
    winner_row = confirmation_rows[0]
    winner_payload = {
        'selection': 'highest internal-validation Dice after Top-K confirmation @256',
        'official_validation_or_test_used': False,
        'trial_number': int(winner_row['trial_number']),
        'confirmation_256_dice': float(winner_row['confirmation_dice']),
        'hpo_128_dice': float(winner_row['hpo_128_dice']),
        'selected_stage': winner_row['selected_stage'],
        'params': winner_row['params'],
        'sampled': winner_row['sampled'],
        'confirmation_checkpoint': winner_row['checkpoint'],
        'run_config_sha256': RUN_CONFIG_SHA256,
    }
    atomic_json_save(winner_payload, winner_path)
elif RUN_CONFIRM_256:
    print(
        'Bỏ qua confirmation trong session này: HPO chưa đạt '
        'TARGET_TOTAL_TRIALS hoặc chưa đủ Top-K. Artifact HPO đã được lưu để resume.'
    )
    winner_payload, confirmation_table = None, pd.DataFrame()
elif winner_path.is_file() and confirmation_results_path.is_file():
    winner_payload = json.loads(winner_path.read_text(encoding='utf-8'))
    confirmation_table = pd.read_csv(confirmation_results_path)
else:
    winner_payload, confirmation_table = None, pd.DataFrame()

CONFIRM_READY = winner_payload is not None
if CONFIRM_READY:
    if winner_payload.get('run_config_sha256') != RUN_CONFIG_SHA256:
        raise RuntimeError('winner.json thuộc protocol/config khác')
    display(confirmation_table.style.format({
        'hpo_128_dice': '{:.4f}', 'confirmation_256_dice': '{:.4f}'
    }))
    print('Locked winner:', winner_payload)
else:
    print('Chưa có winner @256; final CV sẽ không chạy.')


## 3. Final 3-fold với cấu hình winner đã khóa

Winner chỉ cung cấp **hyperparameter**; final CV không dùng checkpoint confirmation. Mỗi fold khởi tạo lại U-Net34 từ ResNet34 ImageNet, dùng fold-train có augmentation và fold-holdout sạch để chọn Best Dice qua bốn stage 128/256. Vì winner đã khóa trước, official validation/test vẫn độc lập với bước tuning.


In [ ]:
def make_folds(records, n_splits, seed):
    permutation = torch.randperm(
        len(records), generator=torch.Generator().manual_seed(int(seed))
    ).tolist()
    fold_sizes = [len(records) // n_splits] * n_splits
    for index in range(len(records) % n_splits):
        fold_sizes[index] += 1
    fold_indices, cursor = [], 0
    for fold_size in fold_sizes:
        fold_indices.append(permutation[cursor:cursor + fold_size])
        cursor += fold_size
    all_indices = set(range(len(records)))
    folds = []
    for holdout_indices in fold_indices:
        holdout_set = set(holdout_indices)
        train_part = [records[i] for i in sorted(all_indices - holdout_set)]
        holdout_part = [records[i] for i in holdout_indices]
        assert {r['image_id'] for r in train_part}.isdisjoint(
            r['image_id'] for r in holdout_part
        )
        folds.append((train_part, holdout_part))
    return folds

def final_fold_contract(fold_number, train_subset, holdout_subset, sampled):
    return {
        'protocol_sha256': RUN_CONFIG_SHA256,
        'phase': 'final_cv',
        'fold': int(fold_number),
        'winner_trial': int(winner_payload['trial_number']),
        'winner_params': winner_payload['params'],
        'sampled': sampled,
        'train_ids_sha256': ids_digest(train_subset),
        'holdout_ids_sha256': ids_digest(holdout_subset),
    }

def train_final_fold(fold_number, train_subset, holdout_subset, sampled):
    fold_dir = FINAL_DIR / f'fold_{fold_number}'
    final_path = fold_dir / 'optimized_best.pt'
    contract = final_fold_contract(
        fold_number, train_subset, holdout_subset, sampled
    )
    contract_sha = config_digest(contract)
    if final_path.is_file():
        payload = trusted_torch_load(final_path, map_location='cpu')
        if payload.get('contract_sha256') != contract_sha:
            raise RuntimeError(f'Final fold checkpoint sai contract: {final_path}')
        print(f'[resume complete] final fold {fold_number}')
        return final_path, payload['metadata']

    has_progress = any(fold_dir.glob('*px_*/progress.pt'))
    set_seed(SEED + 30_000 + fold_number)
    model = to_runtime(UNet34(pretrained=PRETRAINED and not has_progress))
    loss_fn = build_loss(sampled['loss'])
    stage_results = []
    selection_loader = make_loader(
        holdout_subset, 256, train=False,
        batch_size=batch_for_size(sampled['batch_size'], 256), seed=SEED,
    )
    global_best_score = -float('inf')
    global_best_stage = None
    global_best_state = None

    for size_index, size in enumerate((128, 256)):
        batch_size = batch_for_size(sampled['batch_size'], size)
        base_seed = SEED + 30_000 + fold_number * 1000 + size_index * 100

        freeze_first_layer_group(model)
        frozen_contract = {
            **contract, 'size': size, 'stage': 'frozen',
            'epochs': FINAL_FROZEN_EPOCHS,
            'batch_size': batch_size, 'stage_seed': base_seed,
        }
        frozen = fit_stage_resumable(
            model, train_subset, holdout_subset,
            size=size, batch_size=batch_size,
            epochs=FINAL_FROZEN_EPOCHS, freeze_bn=False,
            loss_fn=loss_fn, optimizer_config=sampled['optimizer'],
            stage_seed=base_seed,
            stage_dir=fold_dir / f'{size}px_frozen',
            contract=frozen_contract,
            tag=f'final fold {fold_number} | {size}px | frozen',
        )
        frozen_dice_256 = evaluate_dice(model, selection_loader)
        global_best_score, global_best_stage, global_best_state = update_global_best(
            model, frozen_dice_256, f'{size}px_frozen',
            global_best_score, global_best_stage, global_best_state,
        )
        stage_results.append({
            'size': size, 'stage': 'frozen',
            'selection_dice_256': float(frozen_dice_256), **frozen,
        })

        unfreeze_with_frozen_bn(model)
        unfrozen_contract = {
            **contract, 'size': size, 'stage': 'unfrozen',
            'epochs': FINAL_UNFROZEN_EPOCHS,
            'batch_size': batch_size, 'stage_seed': base_seed + 50,
        }
        unfrozen = fit_stage_resumable(
            model, train_subset, holdout_subset,
            size=size, batch_size=batch_size,
            epochs=FINAL_UNFROZEN_EPOCHS, freeze_bn=True,
            loss_fn=loss_fn, optimizer_config=sampled['optimizer'],
            stage_seed=base_seed + 50,
            stage_dir=fold_dir / f'{size}px_unfrozen',
            contract=unfrozen_contract,
            tag=f'final fold {fold_number} | {size}px | unfrozen',
        )
        unfrozen_dice_256 = evaluate_dice(model, selection_loader)
        global_best_score, global_best_stage, global_best_state = update_global_best(
            model, unfrozen_dice_256, f'{size}px_unfrozen',
            global_best_score, global_best_stage, global_best_state,
        )
        stage_results.append({
            'size': size, 'stage': 'unfrozen',
            'selection_dice_256': float(unfrozen_dice_256), **unfrozen,
        })

    holdout_dice = float(global_best_score)
    if global_best_state is None or global_best_stage is None:
        raise RuntimeError(f'Fold {fold_number}: không tạo được global Best Dice')
    metadata = {
        'fold': int(fold_number),
        'train_count': len(train_subset),
        'holdout_count': len(holdout_subset),
        'train_ids_sha256': ids_digest(train_subset),
        'holdout_ids_sha256': ids_digest(holdout_subset),
        'internal_fold_holdout_dice': float(holdout_dice),
        'selected_stage': global_best_stage,
        'winner_trial': int(winner_payload['trial_number']),
        'winner_params': winner_payload['params'],
        'sampled': sampled,
        'stage_results': stage_results,
        'checkpoint': str(final_path),
    }
    atomic_torch_save({
        'model_state': cpu_state_dict(model),
        'metadata': metadata,
        'contract_sha256': contract_sha,
    }, final_path)
    del global_best_state, model, selection_loader
    gc.collect()
    torch.cuda.empty_cache()
    return final_path, metadata

final_folds_path = METRICS_DIR / 'final_fold_holdout.csv'
fold_checkpoint_paths, fold_metadata = [], []

if RUN_FINAL_CV and CONFIRM_READY:
    winner_sampled = winner_payload['sampled']
    final_folds = make_folds(train_records, N_FINAL_FOLDS, SEED)
    for fold_number, (fold_train, fold_holdout) in enumerate(final_folds, start=1):
        print(
            f'\n===== FINAL FOLD {fold_number}/{N_FINAL_FOLDS}: '
            f'train={len(fold_train)}, holdout={len(fold_holdout)} ====='
        )
        checkpoint_path, metadata = train_final_fold(
            fold_number, fold_train, fold_holdout, winner_sampled
        )
        fold_checkpoint_paths.append(str(checkpoint_path))
        fold_metadata.append(metadata)
    final_fold_table = pd.DataFrame([
        {
            'fold': metadata['fold'],
            'train_count': metadata['train_count'],
            'holdout_count': metadata['holdout_count'],
            'internal_fold_holdout_dice': metadata['internal_fold_holdout_dice'],
            'selected_stage': metadata['selected_stage'],
            'checkpoint': metadata['checkpoint'],
        }
        for metadata in fold_metadata
    ])
    final_fold_table.to_csv(final_folds_path, index=False)
elif RUN_FINAL_CV:
    print('Bỏ qua final CV trong session này: chưa khóa winner @256.')
    final_fold_table = pd.DataFrame()
else:
    expected_paths = [
        FINAL_DIR / f'fold_{fold_number}' / 'optimized_best.pt'
        for fold_number in range(1, N_FINAL_FOLDS + 1)
    ]
    if CONFIRM_READY and all(path.is_file() for path in expected_paths):
        validation_folds = make_folds(train_records, N_FINAL_FOLDS, SEED)
        for fold_number, (path, (fold_train, fold_holdout)) in enumerate(
            zip(expected_paths, validation_folds), start=1
        ):
            payload = trusted_torch_load(path, map_location='cpu')
            expected_contract = final_fold_contract(
                fold_number, fold_train, fold_holdout, winner_payload['sampled']
            )
            if payload.get('contract_sha256') != config_digest(expected_contract):
                raise RuntimeError(
                    f'Final fold checkpoint sai winner/split/config: {path}'
                )
            fold_checkpoint_paths.append(str(path))
            fold_metadata.append(payload['metadata'])
        if final_folds_path.is_file():
            final_fold_table = pd.read_csv(final_folds_path)
        else:
            final_fold_table = pd.DataFrame([
                {
                    'fold': metadata['fold'],
                    'train_count': metadata['train_count'],
                    'holdout_count': metadata['holdout_count'],
                    'internal_fold_holdout_dice': metadata['internal_fold_holdout_dice'],
                    'selected_stage': metadata['selected_stage'],
                    'checkpoint': metadata['checkpoint'],
                }
                for metadata in fold_metadata
            ])
            final_fold_table.to_csv(final_folds_path, index=False)
    else:
        if not CONFIRM_READY:
            print('Không load final checkpoints: thiếu winner.json để kiểm contract.')
        final_fold_table = pd.DataFrame()

FINAL_READY = len(fold_checkpoint_paths) == N_FINAL_FOLDS
print('FINAL_READY=', FINAL_READY)
if FINAL_READY:
    display(final_fold_table.style.format({'internal_fold_holdout_dice': '{:.4f}'}))
else:
    print('Chưa đủ final fold checkpoint; official evaluation sẽ không chạy.')


## 4. Đánh giá official validation và test

Single model được chọn bằng internal fold-holdout Dice trước khi official validation/test được mở. Validation chỉ dùng để báo cáo từng fold và equal-weight ensemble. Test báo cáo single đã khóa và ensemble. Threshold luôn là 0,5; HD95 tính theo pixel trên mask resize 256×256.


In [ ]:
if RUN_OFFICIAL_EVALUATION and FINAL_READY:
    selected_single_index = int(np.argmax([
        metadata['internal_fold_holdout_dice'] for metadata in fold_metadata
    ]))
    selected_single_fold = selected_single_index + 1
    runtime_models = []
    selected_checkpoint_path = FINAL_DIR / 'selected_single_best.pt'

    for index, checkpoint_path in enumerate(fold_checkpoint_paths):
        payload = trusted_torch_load(checkpoint_path, map_location='cpu')
        if index == selected_single_index:
            selected_metadata = dict(payload['metadata'])
            selected_metadata['selection'] = 'highest internal fold-holdout Dice'
            selected_metadata['selected_before_official_evaluation'] = True
            atomic_torch_save({
                'model_state': payload['model_state'],
                'metadata': selected_metadata,
                'contract_sha256': payload['contract_sha256'],
            }, selected_checkpoint_path)
        model = to_runtime(UNet34(pretrained=False))
        load_model_state(model, payload['model_state'])
        runtime_models.append(model)
        del payload
    gc.collect()
    torch.cuda.empty_cache()

    eval_batch_size = MAX_BATCH_256
    validation_loader = make_loader(
        official_val_records, 256, train=False,
        batch_size=eval_batch_size, seed=SEED,
    )
    validation_rows = []
    for fold_index, model in enumerate(runtime_models, start=1):
        validation_rows.append({
            'model': f'BestDice fold {fold_index}',
            **evaluate_full([model], validation_loader, include_hd95=True),
        })
    validation_rows.append({
        'model': 'Equal-weight ensemble',
        **evaluate_full(runtime_models, validation_loader, include_hd95=True),
    })
    validation_table = pd.DataFrame(validation_rows)
    validation_table.to_csv(
        METRICS_DIR / 'official_validation_metrics.csv', index=False
    )
    del validation_loader

    test_loader = make_loader(
        official_test_records, 256, train=False,
        batch_size=eval_batch_size, seed=SEED,
    )
    test_rows = [
        {
            'model': f'Single BestDice fold {selected_single_fold}',
            **evaluate_full(
                [runtime_models[selected_single_index]], test_loader, include_hd95=True
            ),
        },
        {
            'model': 'Equal-weight ensemble',
            **evaluate_full(runtime_models, test_loader, include_hd95=True),
        },
    ]
    test_table = pd.DataFrame(test_rows)
    test_table.to_csv(METRICS_DIR / 'official_test_metrics.csv', index=False)
    del test_loader

    summary = {
        'run_config_sha256': RUN_CONFIG_SHA256,
        'sample_counts': {
            'official_train': len(train_records),
            'hpo_train_subset': len(hpo_train_records),
            'hpo_internal_validation': len(hpo_val_records),
            'official_validation': len(official_val_records),
            'official_test': len(official_test_records),
        },
        'study': {
            'name': study.study_name,
            'terminal_trials': terminal_trial_count(study),
            'state_counts': state_counts,
            'best_trial': int(study.best_trial.number),
            'best_hpo_128_dice': float(study.best_value),
            'storage': str(STUDY_DB),
        },
        'winner': winner_payload,
        'final_fold_checkpoints': fold_checkpoint_paths,
        'final_fold_internal_holdout': final_fold_table.to_dict(orient='records'),
        'selected_single_fold_before_official_evaluation': selected_single_fold,
        'selected_single_checkpoint': str(selected_checkpoint_path),
        'official_validation': validation_table.to_dict(orient='records'),
        'official_test': test_table.to_dict(orient='records'),
        'prediction_threshold': PRED_THRESHOLD,
        'ensemble': 'equal-weight mean probability',
        'hd95_unit': 'pixel at 256x256 resolution',
        'leakage_guard': (
            'Official validation/test were not used by LR Finder, augmentation, '
            'Optuna, pruning, checkpoint selection, threshold selection, or model selection.'
        ),
    }
    summary_path = RUN_DIR / 'summary.json'
    atomic_json_save(summary, summary_path)
    metric_format = {
        column: '{:.4f}' for column in validation_table.columns if column != 'model'
    }
    display(validation_table.style.format(metric_format))
    display(test_table.style.format({
        column: '{:.4f}' for column in test_table.columns if column != 'model'
    }))
    print('Selected single fold:', selected_single_fold)
    print('Selected checkpoint:', selected_checkpoint_path)
    print('Summary:', summary_path)
elif RUN_OFFICIAL_EVALUATION:
    print('Bỏ qua official validation/test trong session này: final CV chưa hoàn tất.')
else:
    print('RUN_OFFICIAL_EVALUATION=False: bỏ qua official validation/test.')


## Artifact và cách resume

Toàn bộ kết quả nằm trong `/kaggle/working/unet34_optuna_1gpu/full_v1` (hoặc `fast_dev_v1`):

- `optuna.db`, `tpe_sampler.pkl`, `trials.csv`, `hpo_topk_128/manifest.json` và Top-K checkpoint.
- `confirmation_256/confirmation_results.csv`, checkpoint của từng candidate và `winner.json`.
- `final_cv/fold_*/optimized_best.pt` cùng `final_cv/selected_single_best.pt`.
- `metrics/final_fold_holdout.csv`, official validation/test CSV, `config.json` và `summary.json`.

Nếu hết thời gian Kaggle giữa pipeline: **Save Version** để giữ output, tạo notebook/session mới, attach output đó làm Input, đặt `RESTORE_FROM` tới thư mục run cũ rồi chạy lại từ đầu. Study tiếp tục đến tổng `TARGET_TOTAL_TRIALS`; confirmation/final stage tiếp tục từ checkpoint gần nhất. Trial Optuna đang dở khi runtime chết sẽ không resume theo epoch, nhưng trial mới sẽ được chạy để đạt target.

Giữ cùng phiên bản Optuna khi resume (phiên bản chính xác nằm trong `config.json`); notebook lưu cả SQLite lẫn RNG state của TPE trong `tpe_sampler.pkl`.

Nếu CUDA báo `illegal memory access`, phải **Restart Session** rồi resume từ artifact. Không tiếp tục gọi `torch.manual_seed()` trong CUDA context đã hỏng; stack trace ở seed thường chỉ là nơi lỗi bất đồng bộ được phát hiện, không phải nguyên nhân gốc.
